# 12.1 - Manual LLM Pipeline (No Framework)
**Phase:** 12 - LangChain / Framework Abstractions
**Status:** VERIFIED
---
## 1. What Are We Solving?
Before touching LangChain we build the exact same pipeline with raw tools: f-strings to build a
prompt, a direct HTTP call to the Groq API with `requests`, manual JSON parsing, and plain Python
logic to glue the steps together. This unit makes every hidden layer of an LLM app visible.
## 2. Why Does This Matter?
When a framework breaks, you fall back to the manual version to diagnose it. Most production LLM
systems keep the critical path framework-free precisely so a live system does not break on a
library upgrade. If you can build it by hand, LangChain stops being magic.
## 3. Prerequisites
- Phase 09 (LLM foundations): API calls, prompt basics
- Python: functions, dicts, f-strings, try/except
## 4. Learning Objectives
By the end of this unit, you should be able to:
- Build a prompt string with an f-string
- Call the Groq HTTP API directly with `requests`
- Manually parse the raw JSON response
- Chain multiple manual steps (classify -> lookup -> answer)
- Enumerate and handle every failure point by hand
## 5. Mental Model
A manual LLM pipeline is a recipe: pick ingredients (data), write the instructions (prompt), send it
to the chef (the LLM API), parse the dish (the JSON response), and serve it (the output).

```text
data -> f-string prompt -> HTTP POST -> JSON response -> parse -> route -> formatted answer
              |                    |                |               |              |
             user input     api.groq.com     try/except     json.loads    lookup table


## 6. Setup: A Raw HTTP Client With a Mock Fallback
We call `https://api.groq.com/openai/v1/chat/completions` directly. When there is no
`GROQ_API_KEY` we return a deterministic mock object so every cell completes offline.

In [1]:
import os, json, requests
from dotenv import load_dotenv
load_dotenv()

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def raw_completion(messages, temperature: float = 0.0, max_tokens: int = 120):
    """POST messages to Groq; return the parsed response dict (never raises)."""
    if not os.environ.get("GROQ_API_KEY"):
        last = messages[-1].get("content", "")
        return {"choices": [{"message": {"role": "assistant", "content": f"mock: {last[:40]}"}}],
                "usage": {"prompt_tokens": 12, "completion_tokens": 6, "total_tokens": 18}}
    try:
        resp = requests.post(
            GROQ_API_URL,
            headers={"Authorization": "Bearer " + os.environ["GROQ_API_KEY"],
                     "Content-Type": "application/json"},
            json={"model": GROQ_MODEL, "messages": messages,
                  "temperature": temperature, "max_tokens": max_tokens},
            timeout=30)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        return {"_error": type(e).__name__}


print("endpoint:", GROQ_API_URL)
print("model:", GROQ_MODEL)
print("key present:", bool(os.environ.get("GROQ_API_KEY")))


endpoint: https://api.groq.com/openai/v1/chat/completions
model: openai/gpt-oss-20b
key present: True


## 7. Building a Prompt With an f-String
A prompt is just a string. No framework needed. We interpolate the user input directly and add a
system line that instructs the model about roles.

In [2]:
def build_prompt(system: str, user: str) -> list:
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]


def sentiment_prompt(text: str) -> list:
    return build_prompt(
        system="Classify sentiment as positive, negative, or neutral. Reply with ONE word only.",
        user=f"Review: '{text}'\nSentiment:",
    )


print(sentiment_prompt("The camera is great but the battery dies fast."))


[{'role': 'system', 'content': 'Classify sentiment as positive, negative, or neutral. Reply with ONE word only.'}, {'role': 'user', 'content': "Review: 'The camera is great but the battery dies fast.'\nSentiment:"}]


## 8. Manually Parsing the JSON Response
The API returns a nested dict. We must walk `choices[0].message.content` ourselves. If anything is
missing we get a clear `KeyError`/`TypeError` — that is an explicit failure point, and learning to
read it is the point of this unit.

In [3]:
def extract_content(response: dict) -> str:
    return response["choices"][0]["message"]["content"]


r = raw_completion(sentiment_prompt("Love this product!"))
print("raw keys:", list(r.keys()))
print("content :", extract_content(r))

# Show the failure point: wrong key / nesting raises loudly.
try:
    extract_content({"nope": True})
except Exception as e:
    print("failure point      :", type(e).__name__, "-", e)


raw keys: ['id', 'object', 'created', 'model', 'choices', 'usage', 'usage_breakdown', 'system_fingerprint', 'x_groq', 'service_tier']
content : positive
failure point      : KeyError - 'choices'


## 9. Every Failure Point Exposed
A raw pipeline can fail in many places. Each of these is handled with a tiny try/except branch,
exactly the boilerplate a framework hides from you.

In [4]:
def robust_parse(raw: dict):
    """Return black-box-style errors instead of crashing."""
    if "_error" in raw:
        return {"ok": False, "reason": f"http-failed: {raw['_error']}"}
    if "choices" not in raw or not raw["choices"]:
        return {"ok": False, "reason": "no choices in response"}
    msg = raw["choices"][0].get("message", {})
    if not msg.get("content"):
        return {"ok": False, "reason": "empty assistant content"}
    return {"ok": True, "text": msg["content"]}


run = raw_completion(sentiment_prompt("Delivery was a disaster."))
parsed = robust_parse(run)
print(parsed)

# Edge case: max_tokens too small can truncate, empty input is meaningless.
print(robust_parse(raw_completion([{"role": "user", "content": ""}])))


{'ok': True, 'text': 'negative'}


{'ok': True, 'text': 'Hello! How can I help you today?'}


## 10. Complete Manual Pipeline: classify -> lookup -> answer
A tiny support system in three manual steps. Step 1 asks the model to classify the ticket into a
category; step 2 looks up a canned answer in a plain dict; step 3 fills the template. No framework,
no abstraction — every step is visible and debuggable.

In [5]:
CATEGORY_ANSWERS = {
    "billing": "Your invoice was emailed to you on the 1st. Contact billing@example.com.",
    "support": "Restart the app, then clear the cache. If it persists, open a ticket.",
    "sales":   "Ask your account manager for the current volume discount table.",
}

reply_man = {
    "billing": "Bill management FAQ",
    "support": "Troubleshooting FAQ",
    "sales":   "Pricing FAQ",
}


def classify(text: str) -> str:
    prompt = sentiment_prompt.__wrapped__ if False else build_prompt(
        system="Classify this support ticket into exactly one of: billing, support, sales. "
               "Return the single word only.",
        user=f"Ticket: {text}",
    )
    ret = robust_parse(raw_completion(prompt))
    if not ret["ok"]:
        return "support"  # safe fallback on any failure
    cat = ret["text"].strip().lower()
    return cat if cat in CATEGORY_ANSWERS else "support"


def lookup(category: str):
    return CATEGORY_ANSWERS.get(category, "We will get back to you shortly.")


def answer(category: str, text: str) -> str:
    reply = lookup(category)
    return f"[Category: {category}] Policy for '{reply_man[category]}' -> {reply}"


def pipeline(text: str) -> str:
    category = classify(text)
    return answer(category, text)


for ticket in ["I was double charged last month!",
               "The app crashes on login.",
               "How much for 500 seats?"]:
    print(pipeline(ticket))


[Category: billing] Policy for 'Bill management FAQ' -> Your invoice was emailed to you on the 1st. Contact billing@example.com.


[Category: support] Policy for 'Troubleshooting FAQ' -> Restart the app, then clear the cache. If it persists, open a ticket.


[Category: sales] Policy for 'Pricing FAQ' -> Ask your account manager for the current volume discount table.




## Common Mistakes

- (3-5 bullets, from roadmap, concrete and specific to the unit)

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| ... | ... | ... |
(row table, 3-5 rows)

## Best Practices

- (3-5 bullets)

## Hands-On Practice

1. **Basic:** ...
2. **Guided:** ...
3. **Independent:** ...
4. **Realistic:** ...
5. **Challenge:** ...

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.


## Notes for This Unit

- The manual helper above is the entire "framework" for this notebook: three small functions.
- Notice how the parsing branch (`robust_parse`) must be repeated everywhere — LangChain
  centralises exactly this in output parsers (Unit 12.3).

### Common Mistakes (applied)

- Not handling HTTP/parse errors — a raw call can fail in 4+ unique ways.
- Hardcoding prompts inline (no version control).
- Assuming the model always emits valid JSON.
- Forgetting `max_tokens` can silently truncate a reply.

### Debugging (applied)

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError: 'choices'` | Non-200 response returned an error body | Inspect `raw` keys, wrap in `robust_parse` |
| Parse fails intermittently | Model added prose around the word | Lower temperature, request one word only |
| Output truncated | `max_tokens` too small | Raise `max_tokens` |
| Horror story: model not called | Missing key/env var | Mock branch prints a `mock:` line |

### Best Practices (applied)

- Always parse behind a helper that returns `ok/False` instead of raising.
- Version-control your system prompts.
- Log the raw response before parsing during development.
- Use `temperature=0.0` for classification tasks.

### Hands-On Practice

1. **Basic:** Rerun with 3 new reviews; read the mock / real output.
2. **Guided:** Add a fourth category (`returns`) and wire it into the table and classifier.
3. **Independent:** Add timeouts and a simple retry (try twice) to `raw_completion`.
4. **Realistic:** Log token usage from `raw["usage"]` and print total cost per ticket.
5. **Challenge:** Build a 3-stage pipeline: extract entity -> classify -> compose a reply in one function.

### Exit Criteria

- You can build a multi-step LLM pipeline with raw API calls.
- You can parse and validate LLM output reliably.
- You can handle common API errors gracefully.
